# BirdCLEF 2026 Training v23 — PerchGRU Sequence Model (Kaggle)
## Bidirectional GRU over per-window Perch embeddings

### Required Kaggle inputs
1. **Competition data** — `birdclef-2026`
2. **Embeddings** — `birdclef-2026-perch-embs-v3` (output of precompute-kaggle-v3)

### Output
Checkpoints saved to `/kaggle/working/` as `perch_gru_v23_fold*.pt`
→ Output tab → New Dataset → name: `birdclef-2026-perch-weights-v23`

In [ ]:
# === CELL 1: IMPORTS & CONFIG ===
import os, json, ast, copy, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

CFG = dict(
    epochs         = 25,
    warmup_epochs  = 4,
    lr             = 5e-4,
    batch_size     = 64,
    patience       = 7,
    num_workers    = 2,     # Kaggle limit
    seed           = 42,
    swa_start_frac = 0.6,
    secondary_label_weight   = 0.3,
    soundscape_sample_weight = 0.8,
    perch_emb_dim  = 1536,
    perch_emb_noise= 0.02,
    gru_hidden     = 512,
    gru_layers     = 2,
    gru_dropout    = 0.3,
    max_seq_len    = 60,
    folds          = 5,
    device         = 'cuda' if torch.cuda.is_available() else 'cpu',
    checkpoint_tag = 'v23',
)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
device = torch.device(CFG['device'])

print(f'v23 — PerchGRU sequence model (Kaggle)')
print(f'   Device  : {device}  (AMP={torch.cuda.is_available()})')
print(f'   Folds   : {CFG["folds"]}  Epochs: {CFG["epochs"]}')
print(f'   GRU     : hidden={CFG["gru_hidden"]}, layers={CFG["gru_layers"]}, bidirectional=True')

In [ ]:
# === CELL 2: PATHS & SPECIES ===
COMP_DIR  = '/kaggle/input/birdclef-2026'
_out_root = '/kaggle/working'

TRAIN_META_CSV  = f'{COMP_DIR}/train.csv'
TAXONOMY_CSV    = f'{COMP_DIR}/taxonomy.csv'
SOUNDSCAPE_ANNO = f'{COMP_DIR}/train_soundscapes_labels.csv'

# Embeddings — try both flat layout and nested layout from precompute output
_embs_candidates = [
    '/kaggle/input/birdclef-2026-perch-embs-v3/perch_embeddings_v3',
    '/kaggle/input/birdclef-2026-perch-embs-v3',
]
EMBD_DIR = next((p for p in _embs_candidates if Path(p).is_dir() and
                 list(Path(p).glob('*.npy'))), None)
if EMBD_DIR is None:
    raise RuntimeError(
        'No embedding .npy files found.\n'
        'Run birdclef2026-perch-precompute-kaggle-v3.ipynb first, then add its '
        'output dataset as input here.'
    )
EMBD_DIR = Path(EMBD_DIR)

_all_emb_files = list(EMBD_DIR.glob('*.npy'))
_focal_stems   = {f.stem for f in _all_emb_files if not f.stem.startswith('soundscape_')}
_sc_stems      = {f.stem for f in _all_emb_files if f.stem.startswith('soundscape_')}

if len(_focal_stems) < 100:
    raise RuntimeError(f'Focal embeddings missing in {EMBD_DIR}. Re-run precompute notebook.')

import random as _rng
_s = np.load(str(_rng.choice(_all_emb_files)))
assert _s.shape == (1536,) and _s.std() > 0.05,     f'Bad embedding: shape={_s.shape}, std={_s.std():.4f}. Re-run precompute notebook.'
print(f'Embedding sanity OK: std={_s.std():.4f}')

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
species_set = set(species)
sp_idx      = {lab: i for i, lab in enumerate(species)}
n_classes   = len(species)
df          = pd.read_csv(TRAIN_META_CSV)

with open(f'{_out_root}/species.json', 'w') as f:
    json.dump(species, f)

print(f'EMBD_DIR          : {EMBD_DIR}')
print(f'  Focal embeds    : {len(_focal_stems)}')
print(f'  Soundscape embs : {len(_sc_stems)}')
print(f'  Species         : {n_classes}')

In [ ]:
# === CELL 3: LABEL HELPERS ===
def parse_secondary(s):
    if pd.isna(s): return []
    t = str(s).strip()
    if t in ('', '[]'): return []
    try:
        lst = ast.literal_eval(t)
        return [str(v) for v in lst] if isinstance(lst, list) else []
    except Exception:
        return []

def row_to_multihot(primary_id: str, secondary_str: str) -> np.ndarray:
    y = np.zeros(n_classes, dtype='float32')
    if str(primary_id) in sp_idx:
        y[sp_idx[str(primary_id)]] = 1.0
    for sid in parse_secondary(secondary_str):
        if sid in sp_idx:
            y[sp_idx[sid]] = CFG['secondary_label_weight']
    return y

def soundscape_to_multihot(label_str: str) -> np.ndarray:
    y = np.zeros(n_classes, dtype='float32')
    for sp in str(label_str).split(';'):
        sp = sp.strip()
        if sp in sp_idx:
            y[sp_idx[sp]] = 1.0
    return y

def _parse_hms(s: str) -> int:
    p = str(s).strip().split(':')
    return int(p[0]) * 3600 + int(p[1]) * 60 + int(p[2])

print('✅ Label helpers defined')

In [ ]:
# === CELL 4: PERCHGRU MODEL ===
class PerchGRU(nn.Module):
    # Bidirectional GRU over a sequence of Perch 1536-d embeddings.
    # Input : (B, T, 1536)  -- T windows per soundscape
    # Output: (B, T, n_classes) -- per-window logits
    def __init__(self, n_classes: int,
                 emb_dim: int = 1536,
                 hidden: int = 512,
                 n_layers: int = 2,
                 dropout: float = 0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 512),
            nn.GELU(),
        )
        self.gru = nn.GRU(
            input_size=512,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(0.2),
            nn.Linear(hidden * 2, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        single = (x.dim() == 2)
        if single:
            x = x.unsqueeze(1)
        z    = self.proj(x)
        h, _ = self.gru(z)
        out  = self.head(h)
        if single:
            out = out.squeeze(1)
        return out

_g = PerchGRU(n_classes, CFG['perch_emb_dim'], CFG['gru_hidden'], CFG['gru_layers'], CFG['gru_dropout'])
print(f'PerchGRU: {sum(p.numel() for p in _g.parameters()) / 1e6:.2f}M parameters')
del _g

In [ ]:
# === CELL 5: SEQUENCE DATASETS ===

class FocalDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, emb_root: Path, train: bool):
        self.df       = frame.reset_index(drop=True)
        self.emb_root = emb_root
        self.train    = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r   = self.df.iloc[i]
        emb = np.load(self.emb_root / (str(r['filename']) + '.npy')).astype('float32')
        if self.train and random.random() < 0.5:
            emb += np.random.randn(*emb.shape).astype('float32') * CFG['perch_emb_noise']
        x = torch.from_numpy(emb).unsqueeze(0)   # (1, 1536)
        y = torch.from_numpy(
            row_to_multihot(r['primary_label'], r.get('secondary_labels', '[]'))
        ).unsqueeze(0).float()                    # (1, n_classes)
        w = torch.tensor(float(r.get('sample_weight', 1.0)))
        return x, y, w


class SoundscapeSeqDataset(Dataset):
    def __init__(self, seq_groups: list, emb_root: Path, train: bool):
        self.groups   = seq_groups
        self.emb_root = emb_root
        self.train    = train

    def __len__(self): return len(self.groups)

    def __getitem__(self, i):
        grp     = self.groups[i]
        windows = sorted(grp['windows'], key=lambda w: w[1])[:CFG['max_seq_len']]
        T       = len(windows)
        embs    = np.zeros((T, CFG['perch_emb_dim']), dtype='float32')
        labels  = np.zeros((T, n_classes), dtype='float32')
        for t, (stem, end_secs, lv) in enumerate(windows):
            ep = self.emb_root / (stem + '.npy')
            if ep.exists():
                e = np.load(str(ep)).astype('float32')
                if self.train and random.random() < 0.5:
                    e += np.random.randn(*e.shape).astype('float32') * CFG['perch_emb_noise']
                embs[t] = e
            labels[t] = lv
        x = torch.from_numpy(embs)    # (T, 1536)
        y = torch.from_numpy(labels)  # (T, n_classes)
        w = torch.tensor(grp.get('weight', 1.0), dtype=torch.float32)
        return x, y, w


def seq_collate(batch):
    xs, ys, ws = zip(*batch)
    max_T  = max(x.shape[0] for x in xs)
    B      = len(xs)
    x_pad  = torch.zeros(B, max_T, CFG['perch_emb_dim'])
    y_pad  = torch.zeros(B, max_T, n_classes)
    mask   = torch.zeros(B, max_T, dtype=torch.bool)
    for i, (x, y) in enumerate(zip(xs, ys)):
        T = x.shape[0]
        x_pad[i, :T] = x
        y_pad[i, :T] = y
        mask[i, :T]  = True
    return x_pad, y_pad, mask, torch.stack(ws)


print('✅ FocalDataset, SoundscapeSeqDataset, seq_collate defined')

In [ ]:
# === CELL 6: BUILD SEQUENCE GROUPS FROM SOUNDSCAPE EMBEDDINGS ===
sc_anno = pd.read_csv(SOUNDSCAPE_ANNO)
_sc_label_map = {}
for _, row in sc_anno.iterrows():
    sc_stem  = Path(str(row['filename'])).stem
    end_secs = _parse_hms(row['end'])
    lv       = soundscape_to_multihot(row['primary_label'])
    _sc_label_map[(sc_stem, end_secs)] = lv

_sc_groups_dict = defaultdict(list)
missing_labels  = 0

for f in EMBD_DIR.glob('soundscape_*.npy'):
    stem_part, end_part = f.stem.rsplit('_', 1)
    if not end_part.endswith('s'):
        continue
    end_secs = int(end_part[:-1])
    sc_stem  = stem_part[len('soundscape_'):]
    lv       = _sc_label_map.get((sc_stem, end_secs))
    if lv is None:
        missing_labels += 1
        continue
    _sc_groups_dict[sc_stem].append((f.stem, end_secs, lv))

seq_groups = [
    {'stem': sc_stem, 'windows': windows, 'weight': CFG['soundscape_sample_weight']}
    for sc_stem, windows in _sc_groups_dict.items()
    if windows
]

focal_df = df.copy()
focal_df['filename'] = focal_df['filename'].apply(lambda x: x.replace('/', '_'))
if 'secondary_labels' not in focal_df.columns:
    focal_df['secondary_labels'] = '[]'
else:
    focal_df['secondary_labels'] = focal_df['secondary_labels'].fillna('[]')
focal_df['sample_weight'] = 1.0
focal_df = focal_df[focal_df['filename'].isin(_focal_stems)].reset_index(drop=True)

print(f'Focal clips          : {len(focal_df)}')
print(f'Soundscape sequences : {len(seq_groups)}')
print(f'  (missing labels    : {missing_labels})')

In [ ]:
# === CELL 7: 5-FOLD TRAINING ===
print('=' * 65)
print(f'v23 PerchGRU Training   {CFG["folds"]} folds  AMP={torch.cuda.is_available()}')
print('=' * 65)

_use_amp   = (device.type == 'cuda')
_criterion = nn.BCEWithLogitsLoss(reduction='none')
skf        = StratifiedKFold(n_splits=CFG['folds'], shuffle=True, random_state=CFG['seed'])

fold_scores = []

for fold_idx, (tr_idx, va_idx) in enumerate(
    skf.split(
        focal_df,
        focal_df['primary_label'].apply(
            lambda x: x if focal_df['primary_label'].value_counts().get(x, 0) >= CFG['folds']
            else '__rare__'
        )
    )
):
    print(f'\nFold {fold_idx + 1}/{CFG["folds"]}')

    focal_tr = focal_df.iloc[tr_idx].reset_index(drop=True)
    focal_va = focal_df.iloc[va_idx].reset_index(drop=True)

    focal_tr_ds = FocalDataset(focal_tr, EMBD_DIR, train=True)
    focal_va_ds = FocalDataset(focal_va, EMBD_DIR, train=False)
    focal_tr_dl = DataLoader(focal_tr_ds, batch_size=CFG['batch_size'],
                             shuffle=True,  num_workers=CFG['num_workers'],
                             collate_fn=seq_collate, drop_last=True,
                             pin_memory=_use_amp)
    focal_va_dl = DataLoader(focal_va_ds, batch_size=CFG['batch_size'],
                             shuffle=False, num_workers=CFG['num_workers'],
                             collate_fn=seq_collate, drop_last=False,
                             pin_memory=_use_amp)

    sc_tr_ds = SoundscapeSeqDataset(seq_groups, EMBD_DIR, train=True)
    sc_tr_dl = DataLoader(sc_tr_ds, batch_size=max(1, CFG['batch_size'] // 4),
                          shuffle=True,  num_workers=CFG['num_workers'],
                          collate_fn=seq_collate, drop_last=True,
                          pin_memory=_use_amp)

    model     = PerchGRU(n_classes, CFG['perch_emb_dim'],
                         CFG['gru_hidden'], CFG['gru_layers'], CFG['gru_dropout']).to(device)
    optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
    scaler    = GradScaler(enabled=_use_amp)
    warmup    = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=CFG['warmup_epochs'])
    cosine    = CosineAnnealingLR(optimizer, T_max=max(1, CFG['epochs'] - CFG['warmup_epochs']), eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[CFG['warmup_epochs']])

    best_auc    = -1.0
    patience_ct = 0
    best_state  = None
    swa_states  = []
    _swa_ep     = max(1, int(CFG['epochs'] * CFG['swa_start_frac']))

    def _compute_loss(logits, y_pad, mask, w):
        loss_per  = _criterion(logits, y_pad)
        m         = mask.unsqueeze(-1).float()
        step_loss = (loss_per * m).sum(dim=[1, 2]) / (m.sum(dim=[1, 2]).clamp(min=1))
        return (step_loss * w).mean()

    for epoch in range(CFG['epochs']):
        model.train()
        train_loss = 0.0
        n_batches  = 0

        sc_iter = iter(sc_tr_dl)
        for xf, yf, mf, wf in focal_tr_dl:
            for src_x, src_y, src_m, src_w in [
                (xf, yf, mf, wf),
                *([next(sc_iter, (None,)*4)])
            ]:
                if src_x is None:
                    continue
                src_x, src_y, src_m, src_w = (
                    src_x.to(device), src_y.to(device),
                    src_m.to(device), src_w.to(device),
                )
                optimizer.zero_grad()
                with autocast(enabled=_use_amp):
                    logits = model(src_x)
                    loss   = _compute_loss(logits, src_y, src_m, src_w)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                train_loss += loss.item()
                n_batches  += 1
        train_loss /= max(n_batches, 1)
        scheduler.step()

        model.eval()
        val_preds, val_targets = [], []
        val_loss = 0.0
        with torch.no_grad():
            for xv, yv, mv, wv in focal_va_dl:
                xv, yv, mv, wv = xv.to(device), yv.to(device), mv.to(device), wv.to(device)
                with autocast(enabled=_use_amp):
                    logits_v = model(xv).float()
                logits_v = logits_v[:, 0, :]
                yv_sq    = yv[:, 0, :]
                val_loss += nn.functional.binary_cross_entropy_with_logits(logits_v, yv_sq).item()
                val_preds.append(torch.sigmoid(logits_v).cpu().numpy())
                val_targets.append(yv_sq.cpu().numpy())
        val_loss /= max(len(focal_va_dl), 1)

        fp = np.vstack(val_preds)   if val_preds   else np.zeros((len(va_idx), n_classes))
        ft = np.vstack(val_targets) if val_targets else np.zeros((len(va_idx), n_classes))
        ft_bin = (ft >= 0.5).astype(np.float32)
        auc_ep = [
            roc_auc_score(ft_bin[:, j], fp[:, j])
            for j in range(n_classes)
            if ft_bin[:, j].sum() > 0 and (1 - ft_bin[:, j]).sum() > 0
        ]
        val_auc = np.mean(auc_ep) if auc_ep else 0.0

        if val_auc > best_auc:
            best_auc    = val_auc
            patience_ct = 0
            best_state  = copy.deepcopy(model.state_dict())
        else:
            patience_ct += 1

        if epoch >= _swa_ep:
            swa_states.append(copy.deepcopy(model.state_dict()))
            if len(swa_states) > 10:
                swa_states.pop(0)

        if (epoch + 1) % 5 == 0 or patience_ct == 0:
            print(f'  Ep {epoch+1:3d}: train={train_loss:.4f}  val={val_loss:.4f}  auc={val_auc:.4f}')

        if patience_ct >= CFG['patience']:
            print(f'  Early stop @ epoch {epoch+1}')
            break

    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())

    if swa_states:
        avg_state = {}
        for k in swa_states[0]:
            avg_state[k] = torch.stack([s[k].float() for s in swa_states]).mean(0).to(swa_states[0][k].dtype)
        model.load_state_dict(avg_state)
        print(f'  SWA: averaged {len(swa_states)} ckpts')
    else:
        model.load_state_dict(best_state)

    ckpt = f'{_out_root}/perch_gru_v23_fold{fold_idx}.pt'
    torch.save(model.state_dict(), ckpt)
    fold_scores.append(best_auc)
    print(f'  Fold {fold_idx+1} best AUC: {best_auc:.4f}  saved {ckpt}')
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'\n✅ Mean OOF AUC: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
print(f'   Fold AUCs  : {[f"{a:.4f}" for a in fold_scores]}')

In [ ]:
# === CELL 8: SUMMARY — SAVED CHECKPOINTS ===
saved = [f for f in Path(_out_root).glob('perch_gru_v23_fold*.pt')]
print(f'Checkpoints saved to /kaggle/working:')
for f in sorted(saved):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')
print()
print('Next steps:')
print('  1. Output tab -> New Dataset -> name: birdclef-2026-perch-weights-v23')
print('  2. Add that dataset as input to birdclef2026-inference-v23-perchonly.ipynb')
print('     and birdclef2026-inference-v24-ensemble.ipynb')